# A researcher workflow with V6
Install the independent `nlpx-termfinder` package into this notebook's Python environment first. This notebook uses only synthetic packaged sources and a fake provider; it makes no paid requests. Fake output verifies software behavior, not model quality. See `docs/PILOT.md` before a separately authorized real collection.

Use a fresh project directory for a new study. The SQLite project persists jobs and evidence across notebook/kernel restarts.

In [ ]:
from pathlib import Path
from importlib.resources import files
import json
from termfinder import Project, FakeProvider
from termfinder.config import starter_recipe, load_recipe

project_path = Path('notebook-cultural-study')
recipe = starter_recipe('cultural')
recipe['mode'] = 'specialists'  # change to 'broad' for the two general passes
recipe['research_question'] = 'Which cultural expressions occur in these synthetic family notes?'
recipe['version'] = '1.0.1'
load_recipe(recipe)
project = Project(project_path)
project.configure(recipe, profile='fake')

The recipe defines inclusion/exclusion policy, original languages, communities, dates, few-shot examples, typed attributes, and finite budgets. The cultural recipe includes established cultural entities. Ordinary customization does not require engine edits. The fake provider recognizes the starter recipe IDs; a custom real project may use its own ID.

In [ ]:
fixture = files('termfinder').joinpath('fixtures', 'cultural.md')
source_ids = project.ingest(paths=[fixture], metadata={
    'language': 'mul', 'community': 'multilingual-community', 'domain': 'synthetic-cultural',
    'constraints': {'redistribution': 'Synthetic demonstration text'},
})
project.estimate()

Review the estimate before collection. Document lengths and specialist routes matter; a count-only estimate is a range, not an expected number of terms. Fake calls have no service bill. A one-job bound simulates an interrupted session without discarding queued work.

In [ ]:
provider = FakeProvider()
project.collect(provider, max_jobs=1)
project.status()

In [ ]:
project = Project(project_path)  # reopening after an interruption/kernel restart
project.resume(FakeProvider())
inventory = project.inventory()
[(term['expression'], term['language'], term['sense']) for term in inventory['terms']]

Inspect evidence against frozen source text. Offsets are half-open Unicode code points, not UTF-8 bytes or JavaScript UTF-16 indices. The expression's occurrence is observed; a proposed meaning or translation remains generated. Grounding does not prove semantic correctness.

In [ ]:
sources = {source['source_id']: source for source in inventory['sources']}
evidence = []
for occurrence in inventory['occurrences']:
    source = sources[occurrence['source_id']]
    observed = source['text'][occurrence['start']:occurrence['end']]
    assert observed == occurrence['expression']
    evidence.append({'observed': observed, 'context': occurrence['context'],
                     'source': source['locator'], 'locations': occurrence.get('source_locations')})
evidence

Optional human review appends an annotation. It does not delete a term from the grounded inventory. Undo keeps the history inspectable. Alias or semantic-merge links work the same way and preserve distinct records.

In [ ]:
term_id = inventory['terms'][0]['term_id']
annotation_id = project.annotate(term_id, 'approved', note='Synthetic source context inspected.')
project.undo_annotation(annotation_id)
project.inventory()['annotations']

In [ ]:
version_id = project.snapshot('notebook-v1')
export_path = project_path / 'exports' / version_id
project.export(version_id, export_path, formats=['csv', 'jsonl', 'parquet', 'annotation'])
print('Frozen version:', version_id)
print('Export directory:', export_path)

The saved version freezes records, policy, prompts, models, source hashes, costs, failures, and coverage. Later collecting or annotating does not change it. Export again from the same version without calling a provider. JSONL retains native types; CSV and Parquet encode nested data as documented in `schema.json`.

For recall evaluation, give qualified annotators `annotation_sources.jsonl` and ask them to read each entire held-out source. They must add all in-scope terms, not merely approve model candidates, then mark `complete: true`. Keep sources/document families together when splitting.

In [ ]:
from termfinder.evaluation import source_splits
split_manifest = source_splits(project.dataset(version_id)['sources'])
split_manifest  # a one-source demo can have empty splits; use a suitable real held-out sample

After exhaustive held-out annotation, load gold packets and run:
```python
from termfinder.evaluation import read_gold, evaluate
report = evaluate(project.dataset(version_id), read_gold('heldout-gold.jsonl'),
                  heldout_source_ids=split_manifest['splits']['test'])
```
The report includes precision, recall, and missed-term evidence by domain/language. Candidate counts and model agreement cannot measure false negatives. Default matching measures expression spans; assess proposed meanings with a separate human sense rubric.